# 20 — Strategy Consultant
> Give it a market name and get back a structured verdict — TAM, competitor breakdown, scored opportunities and risks, and a plain ENTER / MONITOR / AVOID call with rationale.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core langgraph pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
import json
from typing import Literal

from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field


# --- Schema ---

class CompetitorProfile(BaseModel):
    name: str = Field(description="Competitor company name.")
    estimated_market_share_pct: float = Field(
        description="Estimated market share as a percentage (0-100)."
    )
    strengths: list[str] = Field(description="Key competitive advantages.")
    weaknesses: list[str] = Field(description="Notable gaps or vulnerabilities.")


class OpportunityRisk(BaseModel):
    description: str = Field(description="Plain-English description of the opportunity or risk.")
    score: int = Field(description="Severity or attractiveness on a 1-10 scale.", ge=1, le=10)
    category: Literal["opportunity", "risk"] = Field(
        description="Whether this is an opportunity or a risk."
    )


class MarketAnalysis(BaseModel):
    market: str = Field(description="The market or geography being analysed.")
    market_size_usd_bn: float = Field(description="Estimated total addressable market in USD billions.")
    growth_rate_pct: float = Field(description="Annual market growth rate as a percentage.")
    competitors: list[CompetitorProfile] = Field(
        description="Top 3-5 competitors with profile summaries."
    )
    opportunities_and_risks: list[OpportunityRisk] = Field(
        description="Ranked opportunities and risks, highest score first."
    )
    entry_recommendation: Literal["enter", "monitor", "avoid"] = Field(
        description="High-level market entry verdict."
    )
    rationale: str = Field(
        description="Two-to-three sentence plain-English rationale for the entry recommendation."
    )


# --- Research tools ---

_SYSTEM = SystemMessage(
    """You are a strategy consultant. Use the research tools to gather data on the
target market, then return a structured MarketAnalysis.

Steps:
1. Call search_market_size to get TAM and growth rate.
2. Call search_competitors to identify key players.
3. Call search_regulations to understand the regulatory environment.
4. Synthesise all findings into a MarketAnalysis object.

Always base your findings on what the tools return -- do not invent data."""
)


@tool
def search_market_size(market: str) -> str:
    """Return estimated TAM (USD billions) and annual growth rate for a market."""
    data: dict[str, dict] = {
        "b2b saas europe": {"tam_usd_bn": 42.0, "growth_pct": 18.5},
        "industrial iot usa": {"tam_usd_bn": 31.0, "growth_pct": 22.3},
        "default": {"tam_usd_bn": 15.0, "growth_pct": 12.0},
    }
    result = data.get(market.lower(), data["default"])
    return json.dumps(result)


@tool
def search_competitors(market: str) -> str:
    """Return a list of top competitors with market share and positioning."""
    competitors: dict[str, list] = {
        "b2b saas europe": [
            {
                "name": "Salesforce",
                "share_pct": 19.0,
                "strengths": ["brand", "ecosystem"],
                "weaknesses": ["price", "complexity"],
            },
            {
                "name": "HubSpot",
                "share_pct": 11.0,
                "strengths": ["UX", "SMB focus"],
                "weaknesses": ["enterprise gaps"],
            },
            {
                "name": "Pipedrive",
                "share_pct": 5.0,
                "strengths": ["simplicity", "price"],
                "weaknesses": ["limited integrations"],
            },
        ],
        "industrial iot usa": [
            {
                "name": "Siemens MindSphere",
                "share_pct": 14.0,
                "strengths": ["OT expertise", "global reach"],
                "weaknesses": ["cost", "lock-in"],
            },
            {
                "name": "PTC ThingWorx",
                "share_pct": 10.0,
                "strengths": ["developer platform"],
                "weaknesses": ["steep learning curve"],
            },
            {
                "name": "Rockwell FactoryTalk",
                "share_pct": 9.0,
                "strengths": ["installed base"],
                "weaknesses": ["legacy architecture"],
            },
        ],
    }
    default = [{"name": "Incumbent A", "share_pct": 25.0, "strengths": ["scale"], "weaknesses": ["agility"]}]
    result = competitors.get(market.lower(), default)
    return json.dumps(result)


@tool
def search_regulations(market: str) -> str:
    """Return key regulatory considerations for a market and geography."""
    regs: dict[str, str] = {
        "b2b saas europe": (
            "GDPR data residency requirements; NIS2 cybersecurity obligations "
            "for critical sectors; VAT registration per country."
        ),
        "industrial iot usa": (
            "NIST cybersecurity framework compliance; potential FCC spectrum "
            "licensing; sector-specific OSHA safety obligations."
        ),
    }
    return regs.get(
        market.lower(),
        "Standard commercial regulations apply. No sector-specific blockers identified.",
    )


# --- Agent ---

def run(market: str) -> MarketAnalysis:
    """Run a structured market entry analysis for the given market description."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [search_market_size, search_competitors, search_regulations]
    agent = create_react_agent(llm, tools, prompt=_SYSTEM)

    result = agent.invoke({"messages": [{"role": "user", "content": f"Analyse the market: {market}"}]})
    last = result["messages"][-1].content

    structured_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(MarketAnalysis)
    return structured_llm.invoke(
        f"Convert the following research findings into a MarketAnalysis object.\n\nFindings:\n{last}\n\nMarket: {market}"
    )


print("Ready.")

## The scenario

A company is considering entering the Industrial IoT market in the United States — connecting factory equipment to cloud platforms for real-time monitoring and predictive maintenance. Before committing budget, they want to know the market size, who they will be competing against, and whether the regulatory environment is navigable.

The agent calls three research tools in sequence, then delivers a scored breakdown of every opportunity and risk it finds — plus a plain verdict on whether to enter, watch, or walk away.

In [ ]:
market = "Industrial IoT USA"

analysis = run(market)

print(f"{'=' * 60}")
print(f"Market:   {analysis.market}")
print(f"{'=' * 60}")
print(f"TAM:      ${analysis.market_size_usd_bn:.1f}B")
print(f"Growth:   {analysis.growth_rate_pct:.1f}% / year")
print(f"Verdict:  {analysis.entry_recommendation.upper()}")
print(f"\nRationale:\n  {analysis.rationale}")

print(f"\nCompetitors ({len(analysis.competitors)}):")
for c in analysis.competitors:
    strengths = ", ".join(c.strengths)
    weaknesses = ", ".join(c.weaknesses)
    print(f"  {c.name:<26} {c.estimated_market_share_pct:.0f}% share")
    print(f"    + {strengths}")
    print(f"    - {weaknesses}")

print(f"\nOpportunities & Risks ({len(analysis.opportunities_and_risks)}):")
for item in sorted(analysis.opportunities_and_risks, key=lambda x: -x.score):
    tag = "+" if item.category == "opportunity" else "-"
    print(f"  [{tag}] {item.score}/10  {item.description}")

## Use your own data

Replace the input above with:
- `market` — a plain-English description of the market and geography you want to evaluate (e.g. `"HR Tech Southeast Asia"`, `"Healthtech Germany"`, `"Fintech Latin America"`)

The agent will return a TAM estimate, growth rate, competitor breakdown with strengths and weaknesses, a scored list of opportunities and risks, and a clear ENTER / MONITOR / AVOID verdict with rationale.